# 🤖 **Model Benchmark — ETH Target + BTC Cross-Asset + ETH Media**

### 🎯 **Objective**
Evaluate whether combining **BTC cross-asset features** and **ETH media features** improves **ETH crash prediction** beyond any single signal source.

### ⚠️ **Problem Definition**
Binary classification task on **ETH crashes**:

- `True` → ETH Crash (>10% drop within 7 days)
- `False` → No crash

### 🔬 **Benchmark Focus**
Test the **combined signal hypothesis**: BTC market dynamics + media sentiment together may provide better coverage of ETH crash risk than either alone. This is the richest ETH feature set tested.

### 📊 **Feature Set**

| Source | Features | Count |
|---|---|---|
| ETH market (notebook 03 EDA) | 17 standard market features | 17 |
| BTC cross-asset (notebook 07 EDA) | 14 BTC features with cross-asset signal for ETH | 14 |
| ETH media (notebook 05 EDA) | `avg_tone`, `article_count_ma_3`, `tone_ma_7` | 3 |
| **Total** | | **34** |

### 🚀 **Goal**
Determine whether combining cross-asset (BTC) + media (ETH) enrichment produces synergistic improvement over the individual signal sources.

In [ ]:
import pandas as pd
from pipelines.ml_benchmark import run_benchmark_pipeline

### **1. Load ETH, BTC Price and ETH Media Data**

In [ ]:
df_eth = pd.read_csv("../../data/gold/market/eth_usdt_1d_features.csv")
df_btc = pd.read_csv("../../data/gold/market/btc_usdt_1d_features.csv")
df_eth_media = pd.read_csv("../../data/gold/ethereum_tone_gold.csv")

print(f"ETH price : {df_eth.shape[0]} rows, {df_eth.shape[1]} columns")
print(f"BTC price : {df_btc.shape[0]} rows, {df_btc.shape[1]} columns")
print(f"ETH media : {df_eth_media.shape[0]} rows, {df_eth_media.shape[1]} columns")

### **2. Merge ETH + BTC Price (inner join)**

In [ ]:
df_eth["open_time"] = pd.to_datetime(df_eth["open_time"], errors="coerce")
df_btc["open_time"] = pd.to_datetime(df_btc["open_time"], errors="coerce")

btc_renamed = df_btc.rename(columns={c: f"{c}_btc" for c in df_btc.columns if c != "open_time"})

df = pd.merge(df_eth, btc_renamed, on="open_time", how="inner")
df.drop(columns=["target_btc"], inplace=True)

print(f"After ETH+BTC merge : {df.shape[0]} rows, {df.shape[1]} columns")

### **3. Add ETH Media (inner join)**

In [ ]:
df_eth_media["date"] = pd.to_datetime(df_eth_media["date"], errors="coerce")
df_eth_media = (
    df_eth_media
    .dropna(subset=["date"])
    .sort_values("date")
    .drop_duplicates("date", keep="last")
    .reset_index(drop=True)
)

min_date = df["open_time"].min()
max_date = df["open_time"].max()
df_eth_media = df_eth_media[
    (df_eth_media["date"] >= min_date) & (df_eth_media["date"] <= max_date)
].copy()

df = pd.merge(df, df_eth_media, left_on="open_time", right_on="date", how="inner")
df.drop(columns=["open_time", "date"], inplace=True)

print(f"After adding ETH media : {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### **4. Feature Selection**

- **17 ETH market** — from `03_model_benchmark_eth.ipynb`
- **14 BTC cross-features** — from `07_eda_eth_with_btc.ipynb`
- **3 ETH media** — from `05_eda_eth_price_and_media.ipynb`

In [ ]:
SELECTED_FEATURES = [
    # ETH market features (17)
    "return_1d", "return_7d",
    "volatility_7d", "volatility_30d",
    "buy_pressure",
    "drawdown",
    "ma_ratio",
    "lag_return_1d", "lag_return_7d",
    "lag_volatility_7d",
    "lag_buy_pressure",
    "lag_volume_norm",
    "momentum_acc",
    "momentum_volatility",
    "volume",
    "number_of_trades",
    "quote_asset_volume",

    # BTC cross-features (14) — from 07_eda_eth_with_btc.ipynb
    "pressure_x_return_btc", "return_1d_btc", "return_7d_btc", "momentum_volatility_btc",
    "lag_return_1d_btc", "buy_pressure_btc", "momentum_acc_btc", "lag_volume_norm_btc",
    "lag_buy_pressure_btc", "drawdown_btc", "volatility_30d_btc", "lag_return_7d_btc",
    "volatility_7d_btc", "lag_volatility_7d_btc",

    # ETH media enrichment (3) — from 05_eda_eth_price_and_media.ipynb
    "avg_tone",
    "article_count_ma_3",
    "tone_ma_7",
]

TARGET = "target"

df = df[SELECTED_FEATURES + [TARGET]]

print(f"Dataset shape : {df.shape}")
print(f"Target distribution:\n{df[TARGET].value_counts()}")
df.isna().sum().loc[lambda s: s > 0]

### **5. Model Benchmark**

In [ ]:
results = run_benchmark_pipeline(df, verbose=False, plot_confusion=True)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("pr_auc", ascending=False)

# Summary rows
results_df.loc["mean"] = results_df.select_dtypes("number").mean()
results_df.loc["std"]  = results_df.select_dtypes("number").std()

results_df